## UrbanRide_03b — Silver Drivers
**Method:** Batch transformation — Bronze → Silver  
**Input:** `urbanride.bronze.drivers`  
**Output:** `urbanride.silver.drivers`  
**Schedule:** Daily · runs after Urbanride_01 (Bronze CSV ingest)

**Load strategy:**
- First run → full load — processes all Bronze rows, creates Silver table
- Daily runs → incremental — processes only rows where `_ingested_at > last _processed_at`
- Watermark: `_ingested_at` (Bronze) vs `_processed_at` (Silver)
- Target grain: one clean row per `driver_id`

**Transformations applied:**
- Deduplication on `driver_id`
- Rating validation — nullify if outside valid range (3.5–5.0), fill with city average
- Vehicle type validation — flag if not Auto / Sedan / SUV
- Status validation — flag if not active / inactive
- Derived flag: `is_license_expired` — joining_date proxy (no license_expiry in source)
- Partition by `city`

**Note:** Driver city names are clean in the generator — no standardisation needed.

In [0]:
from pyspark.sql.functions import (
    col, when, trim, lower, current_timestamp,
    avg, round, lit, coalesce, broadcast
)
from pyspark.sql.types import DoubleType

CATALOG = 'urbanride'
BRONZE  = f'{CATALOG}.bronze'
SILVER  = f'{CATALOG}.silver'

spark.sql(f'USE CATALOG {CATALOG}')

# ── VALID VALUES ──────────────────────────────────────────────────────────────
VALID_CITIES       = ['Delhi', 'Mumbai', 'Bengaluru', 'Hyderabad', 'Pune']
VALID_VEHICLE_TYPES = ['Auto', 'Sedan', 'SUV']
VALID_STATUSES     = ['active', 'inactive']
RATING_MIN         = 1.0
RATING_MAX         = 5.0

print(f'Catalog : {CATALOG}')
print(f'Source  : {BRONZE}.drivers')
print(f'Target  : {SILVER}.drivers')
print(f'Valid vehicle types : {VALID_VEHICLE_TYPES}')
print(f'Valid statuses      : {VALID_STATUSES}')
print(f'Rating range        : {RATING_MIN} – {RATING_MAX}')
print('Config ready.')


Catalog : urbanride
Source  : urbanride.bronze.drivers
Target  : urbanride.silver.drivers
Valid vehicle types : ['Auto', 'Sedan', 'SUV']
Valid statuses      : ['active', 'inactive']
Rating range        : 1.0 – 5.0
Config ready.


In [0]:
# Check if Silver table already exists
# This determines whether we do a full load or incremental append
print('Checking load type...')

table_exists = spark.catalog.tableExists(f'{SILVER}.drivers')

if not table_exists:
    LOAD_TYPE = 'full'
    last_run  = None
    print('  Silver table does not exist — FULL LOAD')
else:
    LOAD_TYPE = 'incremental'
    last_run  = spark.sql(
        f'SELECT MAX(_processed_at) FROM {SILVER}.drivers'
    ).collect()[0][0]
    print(f'  Silver table exists — INCREMENTAL LOAD')
    print(f'  Last processed at : {last_run}')

print(f'  Load type : {LOAD_TYPE}')


Checking load type...
  Silver table does not exist — FULL LOAD
  Load type : full


In [0]:
print('Reading bronze.drivers...')

df_bronze_full = spark.table(f'{BRONZE}.drivers')

if LOAD_TYPE == 'full':
    df_bronze = df_bronze_full
    print('  Full load — reading all rows')
else:
    df_bronze = df_bronze_full.filter(col('_ingested_at') > last_run)
    print(f'  Incremental load — reading rows ingested after {last_run}')

input_count = df_bronze.count()
print(f'  Rows to process : {input_count:,}')

if input_count == 0:
    print('  No new rows to process — Silver already up to date.')
    dbutils.notebook.exit('No new rows — skipping')

# Profile before cleaning
print()
print('Bronze profile (before cleaning):')
print(f'  Null ratings          : {df_bronze.filter(col("rating").isNull()).count():,}')
print(f'  Ratings out of range  : {df_bronze.filter((col("rating") < RATING_MIN) | (col("rating") > RATING_MAX)).count():,}')
print(f'  Invalid vehicle types : {df_bronze.filter(~col("vehicle_type").isin(VALID_VEHICLE_TYPES)).count():,}')
print(f'  Invalid statuses      : {df_bronze.filter(~col("status").isin(VALID_STATUSES)).count():,}')
print(f'  Duplicate driver_ids  : {input_count - df_bronze.dropDuplicates(["driver_id"]).count():,}')


Reading bronze.drivers...
  Full load — reading all rows
  Rows to process : 20,000

Bronze profile (before cleaning):
  Null ratings          : 0
  Ratings out of range  : 0
  Invalid vehicle types : 0
  Invalid statuses      : 0
  Duplicate driver_ids  : 0


In [0]:
# Deduplicate on driver_id — one row per driver
print('Deduplicating...')

df_deduped = df_bronze.dropDuplicates(['driver_id'])

removed = input_count - df_deduped.count()
print(f'  Rows before : {input_count:,}')
print(f'  Rows after  : {df_deduped.count():,}')
print(f'  Removed     : {removed:,} duplicate rows')


Deduplicating...
  Rows before : 20,000
  Rows after  : 20,000
  Removed     : 0 duplicate rows


In [0]:
print('Validating and filling ratings...')

# Step 1 — flag rows where rating is outside valid range
df_rating = df_deduped.withColumn(
    'is_rating_invalid',
    when(
        col('rating').isNull() |
        (col('rating') < RATING_MIN) |
        (col('rating') > RATING_MAX),
        True
    ).otherwise(False)
)

# Step 2 — nullify invalid ratings (so they can be filled with city avg)
df_rating = df_rating.withColumn(
    'rating',
    when(col('is_rating_invalid') == True, None).otherwise(col('rating'))
)

invalid_count = df_rating.filter(col('is_rating_invalid') == True).count()
print(f'  Invalid ratings nullified : {invalid_count:,}')

# Step 3 — compute city average from valid ratings
# Full load: compute from incoming data
# Incremental: compute from existing Silver to avoid skew from small batch
if LOAD_TYPE == 'full':
    avg_source = df_rating
    print('  Computing city averages from incoming Bronze data (full load)')
else:
    avg_source = spark.table(f'{SILVER}.drivers')
    print('  Computing city averages from existing Silver data (incremental)')

city_avg = avg_source \
    .filter(col('rating').isNotNull()) \
    .groupBy('city') \
    .agg(round(avg('rating'), 2).alias('avg_rating'))

print('  City average ratings:')
city_avg.show()

# Step 4 — fill NULL ratings with city average via broadcast join
df_rating_filled = df_rating \
    .join(broadcast(city_avg), on='city', how='left') \
    .withColumn('rating', coalesce(col('rating'), col('avg_rating'))) \
    .drop('avg_rating')

print(f'  Null ratings remaining : {df_rating_filled.filter(col("rating").isNull()).count()}')


Validating and filling ratings...
  Invalid ratings nullified : 0
  Computing city averages from incoming Bronze data (full load)
  City average ratings:
+---------+----------+
|     city|avg_rating|
+---------+----------+
|   Mumbai|      4.25|
|    Delhi|      4.24|
|     Pune|      4.25|
|Bengaluru|      4.25|
|Hyderabad|      4.25|
+---------+----------+

  Null ratings remaining : 0


In [0]:
print('Validating vehicle type and status...')

# Flag invalid vehicle types
df_validated = df_rating_filled.withColumn(
    'is_vehicle_type_invalid',
    when(~col('vehicle_type').isin(VALID_VEHICLE_TYPES), True).otherwise(False)
)

# Flag invalid statuses
df_validated = df_validated.withColumn(
    'is_status_invalid',
    when(~col('status').isin(VALID_STATUSES), True).otherwise(False)
)

invalid_vtype  = df_validated.filter(col('is_vehicle_type_invalid') == True).count()
invalid_status = df_validated.filter(col('is_status_invalid') == True).count()

print(f'  Invalid vehicle types : {invalid_vtype:,}')
print(f'  Invalid statuses      : {invalid_status:,}')

if LOAD_TYPE == 'full':
    print()
    print('  Vehicle type distribution:')
    df_validated.groupBy('vehicle_type').count().orderBy('count', ascending=False).show()
    print('  Status distribution:')
    df_validated.groupBy('status').count().orderBy('count', ascending=False).show()


Validating vehicle type and status...
  Invalid vehicle types : 0
  Invalid statuses      : 0

  Vehicle type distribution:
+------------+-----+
|vehicle_type|count|
+------------+-----+
|       Sedan| 9020|
|        Auto| 7953|
|         SUV| 3027|
+------------+-----+

  Status distribution:
+--------+-----+
|  status|count|
+--------+-----+
|  active|18996|
|inactive| 1004|
+--------+-----+



In [0]:
# Derive a simple business flag for Gold / ML use
# A high value driver = active + rating >= 4.5 + trips_completed >= 1000
# This mirrors the kind of segmentation Gold will use — computed once here cleanly
print('Deriving is_high_value_driver flag...')

df_flagged = df_validated.withColumn(
    'is_high_value_driver',
    when(
        (col('status') == 'active') &
        (col('rating') >= 4.5) &
        (col('trips_completed') >= 1000),
        True
    ).otherwise(False)
)

high_value_count = df_flagged.filter(col('is_high_value_driver') == True).count()
total_count = df_flagged.count()
pct = high_value_count / total_count * 100
print(f'  High value drivers : {high_value_count:,} ({pct:.1f}%)')
# print(f'  High value drivers : {high_value_count:,} ({round(high_value_count/total_count*100, 1)}%)')


Deriving is_high_value_driver flag...
  High value drivers : 5,657 (28.3%)


In [0]:
df_silver = df_flagged \
    .withColumn('_processed_at', current_timestamp()) \
    .withColumn('_source', lit(f'{BRONZE}.drivers'))

print(f'Final row count   : {df_silver.count():,}')
print(f'Final column count: {len(df_silver.columns)}')
print()
print('Final schema:')
df_silver.printSchema()


Final row count   : 20,000
Final column count: 15

Final schema:
root
 |-- city: string (nullable = true)
 |-- driver_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- vehicle_type: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- trips_completed: integer (nullable = true)
 |-- joining_date: date (nullable = true)
 |-- status: string (nullable = true)
 |-- _ingested_at: timestamp (nullable = true)
 |-- is_rating_invalid: boolean (nullable = false)
 |-- is_vehicle_type_invalid: boolean (nullable = false)
 |-- is_status_invalid: boolean (nullable = false)
 |-- is_high_value_driver: boolean (nullable = false)
 |-- _processed_at: timestamp (nullable = false)
 |-- _source: string (nullable = false)



In [0]:
import time
print(f'Writing silver.drivers — mode: {LOAD_TYPE}...')
t0 = time.time()

if LOAD_TYPE == 'full':
    df_silver.write \
        .format('delta') \
        .mode('overwrite') \
        .partitionBy('city') \
        .option('overwriteSchema', 'true') \
        .saveAsTable(f'{SILVER}.drivers')
    print(f'  Full load written : {df_silver.count():,} rows')
    print(f'  Write time : {time.time()-t0}s')
    print('Running OPTIMIZE...')
    spark.sql(f'OPTIMIZE {SILVER}.drivers')
    print('  OPTIMIZE complete')
else:
    new_count = df_silver.count()
    if new_count > 0:
        df_silver.write \
            .format('delta') \
            .mode('append') \
            .saveAsTable(f'{SILVER}.drivers')
        print(f'  Incremental rows appended : {new_count:,}')
        print(f'  Write time : {time.time()-t0}s')
        print('Running OPTIMIZE...')
        spark.sql(f'OPTIMIZE {SILVER}.drivers')
        print('  OPTIMIZE complete')
    else:
        print('  No new rows — skipping write and OPTIMIZE')


Writing silver.drivers — mode: full...
  Full load written : 20,000 rows
  Write time : 7.600730895996094s
Running OPTIMIZE...
  OPTIMIZE complete


In [0]:
print('=== SILVER DRIVERS VERIFICATION ===')
print()

df_verify = spark.table(f'{SILVER}.drivers')
total = df_verify.count()

print(f'  Total rows : {total:,}')
print(f'  Load type  : {LOAD_TYPE}')
print()

# Data quality checks
print('Data quality checks:')
checks = [
    ('Null driver_id',           df_verify.filter(col('driver_id').isNull()).count()),
    ('Null rating',              df_verify.filter(col('rating').isNull()).count()),
    ('Rating out of range',      df_verify.filter((col('rating') < RATING_MIN) | (col('rating') > RATING_MAX)).count()),
    ('Invalid vehicle type',     df_verify.filter(col('is_vehicle_type_invalid') == True).count()),
    ('Invalid status',           df_verify.filter(col('is_status_invalid') == True).count()),
    ('Null joining_date',        df_verify.filter(col('joining_date').isNull()).count()),
    ('Null trips_completed',     df_verify.filter(col('trips_completed').isNull()).count()),
]

all_passed = True
for check, result in checks:
    status = 'PASS' if result == 0 else 'WARN'
    if result > 0: all_passed = False
    print(f'  {status}  {check:<30} : {result:,}')

print()
print('City distribution:')
df_verify.groupBy('city').count().orderBy('count', ascending=False).show()

print('Vehicle type distribution:')
df_verify.groupBy('vehicle_type').count().orderBy('count', ascending=False).show()

print('Status distribution:')
df_verify.groupBy('status').count().orderBy('count', ascending=False).show()

print('High value driver summary:')
df_verify.groupBy('is_high_value_driver').count().show()

print('Sample rows:')
df_verify.select(
    'driver_id', 'name', 'city', 'vehicle_type',
    'rating', 'trips_completed', 'status',
    'is_rating_invalid', 'is_vehicle_type_invalid',
    'is_status_invalid', 'is_high_value_driver', '_processed_at'
).limit(5).show(truncate=False)

print()
# Delta table details
detail = spark.sql(f'DESCRIBE DETAIL {SILVER}.drivers').collect()[0]
print(f'  numFiles    : {detail["numFiles"]}')
print(f'  sizeInBytes : {detail["sizeInBytes"]:,}')

print()
if all_passed:
    print('All quality checks passed. Silver drivers ready.')
else:
    print('Some checks flagged — review WARN items above.')
    
print('Next: UrbanRide_07 — Gold Customer Features')


=== SILVER DRIVERS VERIFICATION ===

  Total rows : 20,000
  Load type  : full

Data quality checks:
  PASS  Null driver_id                 : 0
  PASS  Null rating                    : 0
  PASS  Rating out of range            : 0
  PASS  Invalid vehicle type           : 0
  PASS  Invalid status                 : 0
  PASS  Null joining_date              : 0
  PASS  Null trips_completed           : 0

City distribution:
+---------+-----+
|     city|count|
+---------+-----+
|    Delhi| 6035|
|   Mumbai| 4953|
|Bengaluru| 3887|
|Hyderabad| 3093|
|     Pune| 2032|
+---------+-----+

Vehicle type distribution:
+------------+-----+
|vehicle_type|count|
+------------+-----+
|       Sedan| 9020|
|        Auto| 7953|
|         SUV| 3027|
+------------+-----+

Status distribution:
+--------+-----+
|  status|count|
+--------+-----+
|  active|18996|
|inactive| 1004|
+--------+-----+

High value driver summary:
+--------------------+-----+
|is_high_value_driver|count|
+--------------------+-----+
| 